<a href="https://colab.research.google.com/github/zxchristina/job-watch/blob/main/HEI_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================================
# HEI BUSINESS HEALTH  |  full EDA + conversion & investment-amount analysis
# One cell, run top to bottom. Sections are labeled; the analysis chunks note
# which exercise question they answer.
# ============================================================================
import pandas as pd, numpy as np
pd.set_option('display.width', 220, 'display.max_columns', 40)

# ---- LOAD: in Colab, UNCOMMENT these two lines and DELETE the FILE line below ----
from google.colab import files
up = files.upload(); FILE = next(iter(up))   # a picker appears; choose the .xlsx
# FILE = "Business_Operations_Analyst_-_Candidate_Exercise_-_Dummy_Dataset.xlsx"
df = pd.read_excel(FILE)

TS = ['OPPORTUNITY_CREATED','APPLICATION_SUBMITTED','APPLICATION_APPROVED','OPPORTUNITY_INVESTED']

# ============================================================================
# SECTION 0  |  BLIND EDA  (start here when you know nothing about the data)
# ============================================================================
print("#"*72); print("# SECTION 0: BLIND EDA -- what IS this dataset?"); print("#"*72)

print(f"\nshape: {df.shape[0]:,} rows x {df.shape[1]} cols")
print("\ncolumns & dtypes:"); print(df.dtypes.to_string())
print("\nfirst 5 rows:"); print(df.head().to_string())
print("\nnumeric summary:"); print(df.describe(include=[np.number]).round(1).to_string())

# What's the grain? (one row per what?)
print(f"\ngrain check -> rows: {len(df):,} | unique OPPORTUNITY_ID: {df.OPPORTUNITY_ID.nunique():,} "
      f"| duplicates: {len(df)-df.OPPORTUNITY_ID.nunique()}")
print("=> one row per opportunity (a single homeowner's journey toward an HEI).")

# Categorical lay of the land
print("\nSTAGE_NAME distribution (the funnel's current-state snapshot):")
print(df.STAGE_NAME.value_counts().to_string())
print(f"\nSTATE: {df.STATE.nunique()} distinct | OPPORTUNITY_NUMBER values: "
      f"{sorted(df.OPPORTUNITY_NUMBER.unique())}")

# Missingness -- and the key insight that it's STRUCTURAL
print("\nmissing % per column:")
print((df.isna().mean()*100).round(2).to_string())
print("=> the 4 timestamps + INVESTMENT_AMOUNT are only filled once an opp reaches")
print("   that milestone, so their null rates just encode how far each opp advanced.")

# Confirm the pipeline order by seeing which timestamps are present at each stage
print("\ntimestamp presence by stage (fraction non-null) -> reveals the funnel order:")
pres = df.groupby('STAGE_NAME')[TS].apply(lambda g: g.notna().mean()).round(2)
pres['N'] = df.STAGE_NAME.value_counts()
print(pres.sort_values('N', ascending=False).to_string())
print("=> pipeline: Created -> Submitted -> Approved -> Invested.")
print("   'Closed Lost' is the catch-all exit; 'In Closing' is approved-but-not-yet-funded.")

# ============================================================================
# CHUNK 1  ->  Q5: data anomalies & caveats
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 1 -> Q5: anomalies & caveats"); print("#"*72)

# Anomaly A: application submitted BEFORE opportunity created (logically impossible)
m = df.OPPORTUNITY_CREATED.notna() & df.APPLICATION_SUBMITTED.notna()
neg = df.loc[m,'APPLICATION_SUBMITTED'] < df.loc[m,'OPPORTUNITY_CREATED']
gap = (df.loc[m,'OPPORTUNITY_CREATED'] - df.loc[m,'APPLICATION_SUBMITTED']).dt.days
print(f"\n[A] submitted-before-created: {neg.sum():,} rows ({100*neg.mean():.2f}% of submitted), "
      f"gap {int(gap[neg].min())}-{int(gap[neg].max())} days (small/bounded -> timestamp quirk).")

# Anomaly B: downstream ordering violations (these SHOULD be zero)
def ord_viol(a,b):
    mm = df[a].notna() & df[b].notna(); return int((df.loc[mm,a] > df.loc[mm,b]).sum())
print(f"[B] submit>approve: {ord_viol('APPLICATION_SUBMITTED','APPLICATION_APPROVED')} | "
      f"approve>invest: {ord_viol('APPLICATION_APPROVED','OPPORTUNITY_INVESTED')} (both should be 0).")

# Range sanity -- catch impossible values
print(f"[C] FICO {df.FICO_SCORE.min():.0f}-{df.FICO_SCORE.max():.0f} (floor=eligibility cutoff) | "
      f"home ${df.HOME_VALUE.min():,.0f}-${df.HOME_VALUE.max():,.0f}")
print(f"[D] data window: {df.OPPORTUNITY_CREATED.min().date()} -> {df[TS].max().max().date()}")

# ============================================================================
# CHUNK 2  ->  foundation: funnel flags + maturity window (censoring guard)
# Recent create-cohorts haven't had time to fund yet (~41d median cycle). If you
# trend raw conversion the last ~2 months look like a cliff: that's RIGHT-CENSORING,
# not a real drop. Build a maturity cutoff so later trends stay honest.
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 2 -> foundation: maturity / censoring guard"); print("#"*72)

# "reached this milestone" booleans: sum()=count reached, mean()=rate reached
df['f_created']=df.OPPORTUNITY_CREATED.notna();   df['f_submitted']=df.APPLICATION_SUBMITTED.notna()
df['f_approved']=df.APPLICATION_APPROVED.notna(); df['f_invested']=df.OPPORTUNITY_INVESTED.notna()

cycle = (df.OPPORTUNITY_INVESTED - df.OPPORTUNITY_CREATED).dt.days
cycle = cycle[cycle >= 0]                       # ignore the anomaly's negative gaps
MATURE_DAYS = int(cycle.quantile(0.99))         # ~63 days = "fully matured" horizon
ASOF = df[TS].max().max()
CUTOFF = ASOF - pd.Timedelta(days=MATURE_DAYS)
df['create_month'] = df.OPPORTUNITY_CREATED.dt.to_period('M')
df['mature_cohort'] = df.OPPORTUNITY_CREATED <= CUTOFF   # True = safe to trend
print(f"\np99 create->invest = {MATURE_DAYS}d | as-of {ASOF.date()} | maturity cutoff {CUTOFF.date()}")
print(f"matured invest rate {df[df.mature_cohort].f_invested.mean()*100:.2f}% "
      f"vs censored tail {df[~df.mature_cohort].f_invested.mean()*100:.2f}% "
      f"(tail artificially low -> exclude from trends).")

# ============================================================================
# CHUNK 3  ->  Q1: conversion, where it stands + how it trended
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 3 -> Q1: conversion level + trend"); print("#"*72)

# Overall funnel across ALL opps. "step" = share of the PREVIOUS milestone that advanced.
c,s,a,i = df.f_created.sum(), df.f_submitted.sum(), df.f_approved.sum(), df.f_invested.sum()
print("\nOverall funnel (cumulative):"); prev=None
for name,v in [("Created",c),("Submitted",s),("Approved",a),("Invested",i)]:
    step=f"{100*v/prev:5.1f}%" if prev else "  -  "
    print(f"  {name:11s} {v:8,d}  {100*v/c:6.2f}% of created   step {step}"); prev=v

# Trend by create-cohort, MATURED cohorts only, so months compare on equal footing.
# NAMING GOTCHA: never name a column 'sub'/'take'/'sum' etc -- they collide with
# pandas methods, so conv.sub returns the METHOD not the column. Defenses: safe
# names (n_submitted) + bracket access conv['n_submitted'].
conv = (df[df.mature_cohort].groupby('create_month')
        .agg(created=('f_created','sum'),
             submit_rate=('f_submitted','mean'),
             n_approved=('f_approved','sum'),      # numerator for approval_rate
             n_submitted=('f_submitted','sum'),    # denominator for approval_rate
             invest_rate=('f_invested','mean')))
conv['submit_rate']   = (conv['submit_rate']*100).round(1)                       # submitted/created
conv['approval_rate'] = (conv['n_approved']/conv['n_submitted']*100).round(1)    # approved|submitted (bottleneck)
conv['invest_rate']   = (conv['invest_rate']*100).round(2)                       # invested/created (end-to-end)
print("\nConversion by MATURED create-cohort (censored tail dropped):")
print(conv[['created','submit_rate','approval_rate','invest_rate']].to_string())
h=len(conv)//2
print(f"\ntrend: invest rate {conv['invest_rate'].iloc[:h].mean():.2f}% (early) "
      f"-> {conv['invest_rate'].iloc[h:].mean():.2f}% (late)")

# ============================================================================
# CHUNK 4  ->  Q1: avg investment amount trend + Q2: what drives it
# amount = home_value * take_rate. Shift-share shows whether the swing is a
# CUSTOMER story (home value mix) or a PRICING story (take rate).
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 4 -> Q1 amount trend + Q2 what drives it"); print("#"*72)

inv = df[df.f_invested & df.HOME_VALUE.notna()].copy()
# take_rate = share of home value invested. DON'T name it 'take' (reserved pd method).
inv['take_rate'] = inv.INVESTMENT_AMOUNT / inv.HOME_VALUE
inv['inv_month'] = inv.OPPORTUNITY_INVESTED.dt.to_period('M')   # trend by CLOSE month

amt = inv.groupby('inv_month').agg(deals=('INVESTMENT_AMOUNT','size'),
        avg_amt=('INVESTMENT_AMOUNT','mean'), avg_home=('HOME_VALUE','mean'),
        avg_take=('take_rate','mean'), med_fico=('FICO_SCORE','median'))
amt['avg_amt']=amt['avg_amt'].round(0); amt['avg_home']=amt['avg_home'].round(0)
amt['avg_take']=(amt['avg_take']*100).round(2)
print("\nAvg investment amount by close (invested) month:")
print(amt.to_string())

# Shift-share: hold one factor at its overall mean, let the other vary by month.
core = inv[inv.inv_month.map(amt['deals']) >= 30]   # drop tiny early months (noisy)
H, T = core.HOME_VALUE.mean(), core.take_rate.mean()
ss = core.groupby('inv_month').agg(home=('HOME_VALUE','mean'), tr=('take_rate','mean'),
                                   amt=('INVESTMENT_AMOUNT','mean'))
ss['cf_home_only'] = ss['home']*T     # counterfactual: only home value moves
ss['cf_take_only'] = H*ss['tr']       # counterfactual: only take rate moves
print("\nSHIFT-SHARE (which factor drives the avg-amount swing?):")
print(f"  swing if only HOME VALUE moved: ${ss['cf_home_only'].max()-ss['cf_home_only'].min():,.0f}")
print(f"  swing if only TAKE RATE moved:  ${ss['cf_take_only'].max()-ss['cf_take_only'].min():,.0f}")
print(f"  actual avg-amount swing:        ${ss['amt'].max()-ss['amt'].min():,.0f}")
print("  => take rate reproduces most of the swing; home value & FICO ~flat -> pricing story.")
print(f"\nTotals: ${inv.INVESTMENT_AMOUNT.sum()/1e6:.1f}M deployed over {len(inv):,} deals "
      f"| median ${inv.INVESTMENT_AMOUNT.median():,.0f}")

# ============================================================================
# CHUNK 5  ->  Q3: segment over/under-performance
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 5 -> Q3: segments"); print("#"*72)

# FICO -- watch the non-obvious result: MIDDLE bands convert best, not the top.
df['fico_band']=pd.cut(df.FICO_SCORE,[0,580,620,660,700,740,780,851],
    labels=['<580','580-619','620-659','660-699','700-739','740-779','780+'],right=False)
sf=df.groupby('fico_band',observed=True).agg(n=('OPPORTUNITY_ID','size'),
    invest_rate=('f_invested','mean'),avg_amt=('INVESTMENT_AMOUNT','mean'))
sf['invest_rate']=(sf['invest_rate']*100).round(1); sf['avg_amt']=sf['avg_amt'].round(0)
print("\nBy FICO band (middle bands convert best):"); print(sf.to_string())

# States (>=3k volume so tiny states don't top the list), best & worst on conversion.
st=df.groupby('STATE').agg(n=('OPPORTUNITY_ID','size'),invest_rate=('f_invested','mean'),
    avg_amt=('INVESTMENT_AMOUNT','mean')); st=st[st['n']>=3000]
st['invest_rate']=(st['invest_rate']*100).round(2); st['avg_amt']=st['avg_amt'].round(0)
st=st.sort_values('invest_rate',ascending=False)
print("\nBy state (vol>=3k), best & worst on conversion:")
print(pd.concat([st.head(4),st.tail(4)]).to_string())

# Home value -- conversion ~flat, but amount scales with home value (a lever).
df['hv_band']=pd.cut(df.HOME_VALUE,[0,200000,350000,500000,750000,1e6,1e10],
    labels=['<200k','200-350k','350-500k','500-750k','750k-1M','1M+'],right=False)
hv=df.groupby('hv_band',observed=True).agg(n=('OPPORTUNITY_ID','size'),
    invest_rate=('f_invested','mean'),avg_amt=('INVESTMENT_AMOUNT','mean'))
hv['invest_rate']=(hv['invest_rate']*100).round(2); hv['avg_amt']=hv['avg_amt'].round(0)
print("\nBy home value band (conversion ~flat; amount scales with home value):")
print(hv.to_string())

# ============================================================================
# CHUNK 6  ->  Q2: WHY conversion sits where it does (velocity + bottleneck)
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 6 -> Q2: velocity / the bottleneck"); print("#"*72)
print()
for x,y,lab in [('OPPORTUNITY_CREATED','APPLICATION_SUBMITTED','create->submit'),
                ('APPLICATION_SUBMITTED','APPLICATION_APPROVED','submit->approve'),
                ('APPLICATION_APPROVED','OPPORTUNITY_INVESTED','approve->invest'),
                ('OPPORTUNITY_CREATED','OPPORTUNITY_INVESTED','create->invest')]:
    d=(df[y]-df[x]).dt.days; d=d[d>=0]
    print(f"  {lab:16s} median {d.median():5.1f}d  mean {d.mean():5.1f}d  p90 {d.quantile(.9):5.1f}d")
print(f"\n  submit {100*s/c:.1f}% | approve|submit {100*a/s:.1f}% (BOTTLENECK: worst conv AND slowest) "
      f"| invest|approve {100*i/a:.1f}%")

# ============================================================================
# CHUNK 7  ->  Q4: recommendations (each tied to a number above)
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 7 -> Q4: recommendations"); print("#"*72)
print("""
1. Attack Submit->Approve: worst conversion AND slowest stage. Biggest combined
   volume + time lever. Dig into decline reasons / underwriting staffing / docs.
2. Focus acquisition on 620-699 FICO and high-convert states (AZ/NV/OH/SC);
   audit high-volume low-convert CA/NY for process or compliance drag.
3. Manage take rate deliberately: it (not home value) drove the avg-amount swing.
   Confirm the early-2026 compression-then-recovery was intentional; set guardrails.
4. Report conversion on MATURED cohorts only; the last ~2 months are censored,
   not declining.""")
print(f"Evidence: approval bottleneck {100*a/s:.1f}% approve|submit | "
      f"FICO 620-699 {sf.loc[['620-659','660-699'],'invest_rate'].mean():.1f}% vs 780+ {sf.loc['780+','invest_rate']:.1f}%")

# ============================================================================
# CHUNK 8  ->  Q6: additional data to gather
# ============================================================================
print("\n"+"#"*72); print("# CHUNK 8 -> Q6: additional data to refine"); print("#"*72)
print("""
- Closed-Lost reason codes + stage-at-loss (explains the approval bottleneck)
- Lead source / channel / marketing spend (conversion + CAC by channel)
- Requested vs approved amount, and pricing-model version (confirm take-rate shifts)
- Underwriting decline reasons, LTV / existing liens, DTI (why approvals fail)
- Timestamps for intermediate stages (AS / UW / Follow-up) for true stage aging
- Rep / team owner, and macro (rates, home-price index) for external context""")
print("\nDONE.")

Saving Business Operations Analyst - Candidate Exercise - Dummy Dataset.xlsx to Business Operations Analyst - Candidate Exercise - Dummy Dataset (2).xlsx
########################################################################
# SECTION 0: BLIND EDA -- what IS this dataset?
########################################################################

shape: 150,000 rows x 11 cols

columns & dtypes:
OPPORTUNITY_ID                    int64
OPPORTUNITY_NUMBER                int64
STAGE_NAME                       object
OPPORTUNITY_CREATED      datetime64[ns]
APPLICATION_SUBMITTED    datetime64[ns]
APPLICATION_APPROVED     datetime64[ns]
OPPORTUNITY_INVESTED     datetime64[ns]
INVESTMENT_AMOUNT               float64
STATE                            object
FICO_SCORE                      float64
HOME_VALUE                      float64

first 5 rows:
   OPPORTUNITY_ID  OPPORTUNITY_NUMBER        STAGE_NAME OPPORTUNITY_CREATED APPLICATION_SUBMITTED APPLICATION_APPROVED OPPORTUNITY_INVESTED  INVEST